# Tutorial: Two-Agent Dormitory Simulation

This notebook walks through a minimal Luvoire simulation with two high-school dormitory roommates over seven days.

## Audience and Goal

Audience: researchers, game designers, and collaborators who want a quick read of Luvoire's core runtime.

Prerequisites: basic Python and the repository installed with `pip install -e .`.

By the end, you can define two personas, run a 7-day social simulation, and inspect action logs, memory, and relationship state.

## Outline

1. Import Luvoire and define a local deterministic responder.
2. Create two personas and a dormitory environment.
3. Run a 7-day simulation with 30-minute ticks.
4. Inspect logs, memories, and relationship graph state.
5. Try one small exercise.

## 1. Setup

The notebook uses `LocalClient` so it runs without API keys. Later phases can swap this for `LLMGateway` with Anthropic/OpenAI providers.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import subprocess
import sys
from collections import Counter
from datetime import datetime

if importlib.util.find_spec('luvoire') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])

from luvoire import Environment, LocalClient, Persona, Personality, Simulator

## 2. Define Personas

Alice is introverted and night-oriented. Bob is extroverted and morning-oriented. The contrast creates small coordination pressure around shared space.

In [ ]:
alice = Persona(
    agent_id='alice',
    name='Alice Kim',
    age=17,
    background='Introverted literature student in a Korean high-school dormitory; prefers quiet late-night writing.',
    personality=Personality(
        openness=0.82,
        conscientiousness=0.62,
        extraversion=0.22,
        agreeableness=0.68,
        neuroticism=0.46,
    ),
    values=['privacy', 'honesty', 'calm routines'],
    goals=['finish a short story', 'avoid unnecessary conflict'],
)

bob = Persona(
    agent_id='bob',
    name='Bob Park',
    age=17,
    background='Extroverted science student in the same dormitory; wakes early and likes discussing ideas aloud.',
    personality=Personality(
        openness=0.64,
        conscientiousness=0.76,
        extraversion=0.88,
        agreeableness=0.72,
        neuroticism=0.28,
    ),
    values=['curiosity', 'teamwork', 'clear schedules'],
    goals=['prepare for physics contest', 'keep a friendly room atmosphere'],
)

agents = [alice, bob]
[agent.to_system_prompt().splitlines()[0] for agent in agents]

## 3. Deterministic Local Responder

This function imitates an LLM response in strict JSON. It keeps the demo reproducible and fast.

In [ ]:
turns = Counter()


def dorm_responder(messages):
    system_prompt = messages[0]['content']
    agent_id = 'alice' if 'Persona ID: alice' in system_prompt else 'bob'
    turns[agent_id] += 1
    day = ((turns[agent_id] - 1) // 48) + 1

    if agent_id == 'alice':
        content = f'Day {day}: I can keep the desk quiet after dinner if you need the morning slot.'
        target = 'bob'
    else:
        content = f'Day {day}: I will write the room schedule down so we both know when to study.'
        target = 'alice'

    return json.dumps(
        {'action_type': 'speak', 'target': target, 'content': content},
        ensure_ascii=False,
    )

local_llm = LocalClient(dorm_responder)

## 4. Run the Simulation

Seven days with 30-minute ticks produces 336 ticks. With two agents, that is 672 action records.

In [ ]:
environment = Environment(
    start_time=datetime(2026, 3, 2, 9, 0),
    location_path=('Korea', 'Seoul', 'High School Dormitory', 'Room 201'),
    conditions={'semester': 'spring', 'weather': 'light rain', 'exam_week': False},
)

sim = Simulator(
    agents=agents,
    environment=environment,
    tick_duration_minutes=30,
    llm=local_llm,
)

logs = sim.run(duration_days=7)
len(logs), logs[0].timestamp, logs[-1].timestamp

## 5. Inspect Action Logs

The log is JSONL-ready, so it can feed later dashboard and replay tools.

In [ ]:
sample = [
    {
        'tick': entry.tick,
        'time': entry.timestamp.isoformat(),
        'agent': entry.agent_id,
        'target': entry.action.target,
        'content': entry.action.content,
    }
    for entry in logs[:6]
]
sample

## 6. Inspect Relationship State

Repeated neutral coordination increases familiarity and relationship weight while keeping trust stable.

In [ ]:
relationship_graph = sim.relationships.to_networkx()
edge_rows = []
for source, target, data in relationship_graph.edges(data=True):
    edge_rows.append(
        {
            'source': source,
            'target': target,
            'type': data['relationship_type'],
            'weight': round(data['weight'], 3),
            'trust': round(data['trust'], 3),
            'familiarity': round(data['familiarity'], 3),
        }
    )
edge_rows

## 7. Inspect Short-Term Memory

Each agent keeps the latest local actions in a FIFO short-term memory buffer.

In [ ]:
memory_snapshot = {
    agent_id: [memory.content for memory in buffer.recent(3)]
    for agent_id, buffer in sim.short_term_memories.items()
}
memory_snapshot

## Exercise

Change Alice's `extraversion` to `0.7`, rerun the notebook, and compare whether your local responder should produce more direct dialogue.

Answer scaffold: update Alice's `Personality(...)`, then rerun cells 2 through 7. In a real LLM-backed run, this should affect prompt-conditioned behavior.

## Pitfall and Extension

Pitfall: if a notebook silently uses previous state, results become misleading. Always rerun from the first cell before sharing.

Extension: replace `LocalClient` with `LLMGateway` and a real provider once API credentials are available.